# Ex.No 8 — Implement Type Checking using LEX and YACC


## AIM
To write a program using FLEX and BISON to implement type checking of variables in simple declarations and expressions, using a symbol table built during parsing.


## ALGORITHM / PROCEDURE
1. Use FLEX to tokenize keywords (`int`, `float`), identifiers, numbers and operators, passing them to BISON.
2. In BISON, define grammar rules for declaration statements and assignment statements.
3. On a declaration (e.g. `int a;`), insert the variable name and its type into the symbol table.
4. On an assignment (e.g. `a = b + c;`), look up the types of the result variable and operands in the symbol table.
5. If a variable used is not found in the symbol table, report it as undefined.
6. If all types match, print "No type mismatch"; otherwise print "Type mismatch".
7. End the program.

**Procedure**
1. Create `typecheck.l`: define patterns for keywords, identifiers, numbers, operators; return tokens to BISON.
2. Create `typecheck.y`: define grammar rules for declarations and expressions; maintain a symbol table (name/type pairs) filled in on each declaration; on each assignment verify operand/result types.
3. Compile: `flex typecheck.l` → `bison -d typecheck.y` → `gcc lex.yy.c typecheck.tab.c -o typecheck -lfl`.
4. Run `./typecheck`, input declarations and expressions ending with `;`, and observe the type-checking result. Test correctly typed expressions, mismatched types, and undeclared variables.


## PSEUDOCODE / LOGIC
```
DECLARE symbol_table[]                       // array of {name, type}

FUNCTION insert(name, type): APPEND {name, type} to symbol_table
FUNCTION typeOf(name):
    FOR each entry in symbol_table:
        IF entry.name == name THEN RETURN entry.type
    RETURN "undefined"

GRAMMAR:
    decl   -> INT ID ';'    { insert(ID, "int") }
            | FLOAT ID ';'  { insert(ID, "float") }
    assign -> ID '=' expr ';'
            lt = typeOf(ID)
            IF lt == "undefined"      -> PRINT "Undefined variable: " + ID
            ELSE IF lt == type(expr)  -> PRINT "No type mismatch in expression: " + ID + " = ..."
            ELSE                       -> PRINT "Type mismatch in assignment to " + ID
    expr   -> ID   { type = typeOf(ID); if undefined -> report }
            | NUM  { type = "int" }
            | expr op expr { type = (types match) ? that type : "mismatch" }

BEGIN
    CALL yyparse() over declarations and assignments
END
```


## PROGRAM & OUTPUT
The cells below contain the source program (FLEX/BISON/C) and its executed output.


In [82]:
# ============================================================
# TYPE CHECKING USING LEX AND YACC
# GOOGLE COLAB - COMPLETE SINGLE CELL
# ============================================================

# Install required packages
!apt-get update -qq
!apt-get install -y flex bison gcc -qq


# -------------------- typecheck.l --------------------

with open("typecheck.l", "w") as f:
    f.write(r'''
%{
#include "typecheck.tab.h"
#include <string.h>
#include <stdlib.h>
%}

%option noyywrap

%%

"int"       { return INT; }
"float"     { return FLOAT; }

[a-zA-Z_][a-zA-Z0-9_]* {
    yylval.str = strdup(yytext);
    return ID;
}

[0-9]+ {
    yylval.str = strdup(yytext);
    return NUM;
}

"="         { return '='; }
"+"         { return '+'; }
"-"         { return '-'; }
"*"         { return '*'; }
"/"         { return '/'; }
";"         { return ';'; }

[ \t\n]+    { }

.           { return yytext[0]; }

%%
''')


# -------------------- typecheck.y --------------------

with open("typecheck.y", "w") as f:
    f.write(r'''
%{
#include <stdio.h>
#include <stdlib.h>
#include <string.h>

struct sym
{
    char name[20];
    char type[10];
};

struct sym table[50];
int n = 0;

void insert(char *name, char *type)
{
    strcpy(table[n].name, name);
    strcpy(table[n].type, type);
    n++;
}

char *typeOf(char *name)
{
    int i;

    for (i = 0; i < n; i++)
    {
        if (strcmp(table[i].name, name) == 0)
            return table[i].type;
    }

    return "undefined";
}

int yylex(void);
int yyerror(char *s);
%}

%union
{
    char *str;
}

%token <str> ID NUM
%token INT FLOAT

%type <str> expr

%left '+' '-'
%left '*' '/'

%%

program:
    stmts
    ;

stmts:
      stmts stmt
    | stmt
    ;

stmt:
      decl
    | assign
    ;

decl:
      INT ID ';'
      {
          insert($2, "int");
      }

    | FLOAT ID ';'
      {
          insert($2, "float");
      }
    ;

assign:
    ID '=' expr ';'
    {
        char *lt = typeOf($1);

        if (strcmp(lt, "undefined") == 0)
        {
            printf("Undefined variable: %s\n", $1);
        }
        else if (strcmp(lt, $3) == 0)
        {
            printf("No type mismatch in expression: %s = ...\n", $1);
        }
        else
        {
            printf("Type mismatch in assignment to %s\n", $1);
        }
    }
    ;

expr:
      ID
      {
          $$ = typeOf($1);
      }

    | NUM
      {
          $$ = "int";
      }

    | expr '+' expr
      {
          if (strcmp($1, $3) == 0)
              $$ = $1;
          else
              $$ = "mismatch";
      }

    | expr '-' expr
      {
          if (strcmp($1, $3) == 0)
              $$ = $1;
          else
              $$ = "mismatch";
      }

    | expr '*' expr
      {
          if (strcmp($1, $3) == 0)
              $$ = $1;
          else
              $$ = "mismatch";
      }

    | expr '/' expr
      {
          if (strcmp($1, $3) == 0)
              $$ = $1;
          else
              $$ = "mismatch";
      }
    ;

%%

int main()
{
    printf("Enter declarations and expressions:\n");
    yyparse();
    return 0;
}

int yyerror(char *s)
{
    printf("Syntax Error: %s\n", s);
    return 0;
}
''')


# Remove old generated files
!rm -f typecheck.tab.c typecheck.tab.h lex.yy.c typecheck


# Generate BISON and FLEX files
!bison -d typecheck.y
!flex typecheck.l


# Compile
!gcc lex.yy.c typecheck.tab.c -o typecheck -lfl


# -------------------- INPUT --------------------

with open("input.txt", "w") as f:
    f.write("""int a;
int b;
int c;
a = b * c;
""")


# -------------------- RUN --------------------

import subprocess

result = subprocess.run(
    ["./typecheck"],
    stdin=open("input.txt", "r"),
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

print(result.stdout)

Enter declarations and expressions:
No type mismatch in expression: a = ...



## RESULT
Thus, the FLEX and BISON program for type checking was successfully implemented. The program builds a symbol table from declarations and checks type consistency in assignment expressions.
